# 02 · 채점 — "맞았다"를 숫자로

01에서 지문(대화)과 시험 문제(QA)를 봤다. 시험을 칠 학생(memory 시스템)은
아직 없다 — 03에서 만든다. 그런데 왜 채점부터 하냐면:
**채점기는 LLM이 필요 없어서 공짜고, 학생이 생기는 순간 바로 점수를 매길 수
있어야 하기 때문이다.** 그때까지 시스템의 답은 가상으로 지어낸다.

채점 규칙은 MemoryOS 논문을 따른다:

> "On the LoCoMo benchmark, standard **F1** and **BLEU-1** are employed" — 논문 원문

## F1 — 겹침 점수

소문자로 바꾸고, 단어로 쪼개고, 겹침을 잰다.
군더더기를 말하면(precision) 깎이고, 정답 단어를 빠뜨리면(recall) 깎인다.

In [1]:
from memlab.data import load_locomo
from memlab.evaluation import bleu1, set_f1, standard_f1

qa = load_locomo()[0].qa[82]  # 01에서 봤던 시험 문제
reference = qa.answer
prediction = "It raised awareness for mental health issues."  # 가상의 시스템 답

print(f"문제:   {qa.question}")
print(f"정답:   {reference}")
print(f"시스템: {prediction}")
print(f"F1 = {set_f1(prediction, reference):.2f}")

문제:   What did the charity race raise awareness for?
정답:   mental health
시스템: It raised awareness for mental health issues.
F1 = 0.44


**내용은 맞았는데 0.44점.** 길게 말한 만큼 precision이 깎였다.
F1은 "정답 단어만, 군더더기 없이"를 보상한다 —
벤치마크의 답변 프롬프트가 늘 "간결하게 답하라"고 하는 이유다.

## 원본 코드의 F1은 조금 다르다

논문은 "standard F1"이라고 썼지만, 원본 repo 코드는 단어를 **set에 넣는다** —
같은 단어의 반복이 무시된다. 반복이 있는 답에서만 두 방식이 갈린다:

In [2]:
prediction = "health health health"
print(f"원본 방식 set_f1:  {set_f1(prediction, reference):.2f}   ← 반복을 못 봄")
print(f"표준 standard_f1:  {standard_f1(prediction, reference):.2f}   ← 반복을 벌함")

원본 방식 set_f1:  0.67   ← 반복을 못 봄
표준 standard_f1:  0.40   ← 반복을 벌함


그래서 우리 채점기는 **둘 다** 계산한다.
원본과 비교할 땐 `set_f1`, 다른 논문과 비교할 땐 `standard_f1`.

## BLEU-1 — 정밀도 + 짧은 답 벌점

BLEU-1은 recall이 없다. 벌점이 없다면 한 단어만 맞혀도 만점이다.
그래서 정답보다 짧은 답은 exp(1 − 정답길이/답길이)로 깎는다:

In [3]:
long_answer = "It raised awareness for mental health issues."
print(f"bleu1('mental health')  = {bleu1('mental health', reference):.2f}")
print(f"bleu1('mental')         = {bleu1('mental', reference):.2f}   ← 정밀도 1.0이지만 벌점 e^-1")
print(f"bleu1(긴 답)            = {bleu1(long_answer, reference):.2f}   ← 벌점은 없지만 정밀도 2/7")

bleu1('mental health')  = 1.00
bleu1('mental')         = 0.37   ← 정밀도 1.0이지만 벌점 e^-1
bleu1(긴 답)            = 0.29   ← 벌점은 없지만 정밀도 2/7


## cat5(adversarial)는 채점이 뒤집힌다

cat5는 정답이 없다("모른다"가 정답). 그런데 원본 실험 코드는 결과를 저장할 때
**정답 칸에 함정 오답(adversarial_answer)을 넣는다.** 그 상태로 F1을 계산하면:

- 시스템이 잘 버티면 ("I don't know") → 함정 답과 안 겹침 → **0점**
- 시스템이 속아서 지어내면 → 함정 답과 겹침 → **높은 점수**

즉 **cat5는 F1이 높을수록 나쁘다** (환각을 잰 것).
cat1~4와 절대 한 평균에 섞으면 안 되고, 우리 채점 리포트는 cat5를 따로 표시한다.

## 어휘 지표의 한계

F1/BLEU는 **단어가 겹치는지**만 본다. 뜻은 모른다:

In [4]:
# 뜻은 같은데 감점
print(f"'May 7th, 2023' vs '7 May 2023'  → F1 {set_f1('May 7th, 2023', '7 May 2023'):.2f}")
# 뜻은 틀렸는데 득점
print(f"'not mental health' vs 'mental health' → F1 {set_f1('not mental health', 'mental health'):.2f}")

'May 7th, 2023' vs '7 May 2023'  → F1 0.67
'not mental health' vs 'mental health' → F1 0.80


같은 날짜인데 `7th ≠ 7`이라 감점되고, 뜻이 정반대인 답(`not ...`)이 고득점한다.

이 약점 때문에 최근 논문들(Mem0, LongMemEval 등)은 **LLM-as-a-Judge**
(LLM에게 "이 답 맞아?"를 묻기)를 병용한다. 대신 비용이 들고, judge 판정이
흔들려서 재현성이 나빠진다.

**우리 선택**: MemoryOS 논문과 같은 자(F1/BLEU-1)로 baseline을 재고,
내 메소드 실험 때 필요하면 judge를 추가한다. 두 가지 실용 교훈:

1. 답변 프롬프트의 형식 지시(날짜 표기 등)가 점수를 흔든다 → **원본 프롬프트 그대로**
2. 지표가 단어 겹침이므로, 점수 차이를 해석할 땐 실제 답을 몇 개 눈으로 볼 것

## 1,986개를 한 번에 — 배치 채점기

실제로 시스템이 만들 결과물은 (문제, 답) 쌍 1,986개다.
`score()`가 그걸 통째로 받아 카테고리별 리포트를 낸다.

**정답을 그대로 답하는 가상의 만점 학생**으로 리포트 모양을 확인해 보자:

In [5]:
from memlab.evaluation import format_report, score

samples = load_locomo()
pairs = [(q, q.answer or "I don't know") for s in samples for q in s.qa]
print(format_report(score(pairs)))

category      |    n | set_f1 | std_f1 | bleu1
--------------+------+--------+--------+------
MULTI_HOP     |  282 | 1.0000 | 1.0000 | 1.0000
TEMPORAL      |  321 | 1.0000 | 1.0000 | 1.0000
OPEN_DOMAIN   |   96 | 1.0000 | 1.0000 | 1.0000
SINGLE_HOP    |  841 | 1.0000 | 1.0000 | 1.0000
ADVERSARIAL   |  446 | 0.0053 | 0.0053 | 0.0047  (높을수록 나쁨)
--------------+------+--------+--------+------
OVERALL(1~4)  | 1540 | 1.0000 | 1.0000 | 1.0000


만점 학생이니 cat1~4는 전부 1.0, cat5만 0 근처다 — cat5는 함정 답 기준이라 **0이 좋은 것**.

03부터는 이 표의 숫자가 진짜 시스템의 실력으로 채워진다.

## 정리

- **F1** = 군더더기 벌점(precision) × 빠뜨림 벌점(recall)의 조화평균
- 원본 코드는 set 방식(반복 무시) — 논문의 "standard F1"과 다르다 → 둘 다 계산
- **BLEU-1** = 정밀도 + 짧은 답 벌점
- cat5는 함정 답 기준이라 **높을수록 나쁨** — overall(cat1~4)에서 제외
- `score()`가 1,986쌍을 받아 리포트를 낸다 — 시스템이 생기면 바로 물릴 수 있는 상태
- 채점이 원본과 같다는 건 차분 테스트가 보증한다 (`tests/test_metrics.py`)

**다음 — 03 · MemoryOS 해부**: 학생(memory 시스템)이 실제로 뭘 하는지 —
저장하고, 버리고, 찾아서 답하는 구조를 원본 코드로 읽는다.